In [2]:
!pip install requests pandas numpy

In [4]:
"""
Federal Contract Spend Analysis
Step 1: Pull data from USAspending.gov API
Author: Kesheka Edupuganti
"""

import requests
import pandas as pd
import json
import time
import os
from datetime import datetime

# ── Config ──────────────────────────────────────────────────────────────────
BASE_URL = "https://api.usaspending.gov/api/v2"
OUTPUT_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FISCAL_YEAR = 2024  # Change to pull different years

# Top agencies to profile (CGAC codes)
TARGET_AGENCIES = {
    "097": "Department of Defense",
    "075": "Department of Health and Human Services",
    "047": "General Services Administration",
    "089": "Department of Energy",
    "021": "Department of Agriculture",
    "070": "Department of Homeland Security",
}

# NAICS sectors of interest
TARGET_NAICS_PREFIXES = [
    "54",   # Professional, Scientific & Technical Services
    "33",   # Manufacturing
    "23",   # Construction
    "62",   # Health Care & Social Assistance
    "51",   # Information Technology
    "48",   # Transportation
]


# ── Helpers ──────────────────────────────────────────────────────────────────
def safe_get(url, params=None, retries=3, delay=1.5):
    """GET with retry logic."""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    return None


def safe_post(url, payload, retries=3, delay=1.5):
    """POST with retry logic."""
    for attempt in range(retries):
        try:
            r = requests.post(url, json=payload, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    return None


# ── Pull 1: Agency Spend Summary ─────────────────────────────────────────────
def pull_agency_spend():
    print("\n[1/3] Pulling agency spend summaries...")
    records = []

    for cgac, name in TARGET_AGENCIES.items():
        print(f"  → {name}")
        url = f"{BASE_URL}/agency/{cgac}/budgetary_resources/"
        data = safe_get(url, params={"fiscal_year": FISCAL_YEAR})

        if data and "agency_data_by_year" in data:
            for yr_block in data["agency_data_by_year"]:
                if yr_block.get("fiscal_year") == FISCAL_YEAR:
                    records.append({
                        "cgac_code": cgac,
                        "agency_name": name,
                        "fiscal_year": FISCAL_YEAR,
                        "total_budgetary_resources": yr_block.get("agency_budgetary_resources", 0),
                        "total_obligations": yr_block.get("agency_total_obligated", 0),
                        "total_outlays": yr_block.get("agency_total_outlays", 0),
                    })
        time.sleep(0.5)

    df = pd.DataFrame(records)
    path = os.path.join(OUTPUT_DIR, "agency_spend.csv")
    df.to_csv(path, index=False)
    print(f"  Saved {len(df)} agency records → {path}")
    return df


# ── Pull 2: Contract Awards by NAICS ─────────────────────────────────────────
def pull_naics_awards():
    print("\n[2/3] Pulling contract awards by NAICS...")
    all_records = []

    for prefix in TARGET_NAICS_PREFIXES:
        print(f"  → NAICS prefix {prefix}...")
        payload = {
            "filters": {
                "time_period": [{"start_date": f"{FISCAL_YEAR - 1}-10-01",
                                  "end_date":   f"{FISCAL_YEAR}-09-30"}],
                "award_type_codes": ["A", "B", "C", "D"],  # Contracts only
                "naics_codes": [prefix],
            },
            "fields": [
                "Award ID", "Recipient Name", "Award Amount",
                "NAICS Code", "NAICS Description", "Awarding Agency",
                "Period of Performance Start Date", "Period of Performance Current End Date",
                "Place of Performance State Code",
            ],
            "page": 1,
            "limit": 100,
            "sort": "Award Amount",
            "order": "desc",
        }

        data = safe_post(f"{BASE_URL}/search/spending_by_award/", payload)
        if data and "results" in data:
            for row in data["results"]:
                row["naics_sector_prefix"] = prefix
                all_records.append(row)

        time.sleep(0.8)

    df = pd.DataFrame(all_records)
    path = os.path.join(OUTPUT_DIR, "naics_awards_raw.csv")
    df.to_csv(path, index=False)
    print(f"  Saved {len(df)} award records → {path}")
    return df


# ── Pull 3: Award Size Distribution ──────────────────────────────────────────
def pull_award_distribution():
    print("\n[3/3] Pulling award size distribution...")
    payload = {
        "filters": {
            "time_period": [{"start_date": f"{FISCAL_YEAR - 1}-10-01",
                              "end_date":   f"{FISCAL_YEAR}-09-30"}],
            "award_type_codes": ["A", "B", "C", "D"],
        },
        "category": "awarding_agency",
        "limit": 50,
        "page": 1,
    }

    data = safe_post(f"{BASE_URL}/search/spending_by_category/", payload)
    records = []

    if data and "results" in data:
        for item in data["results"]:
            records.append({
                "agency": item.get("name", ""),
                "amount": item.get("amount", 0),
                "fiscal_year": FISCAL_YEAR,
            })

    df = pd.DataFrame(records)
    path = os.path.join(OUTPUT_DIR, "award_distribution.csv")
    df.to_csv(path, index=False)
    print(f"  Saved {len(df)} distribution records → {path}")
    return df


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"=== USAspending Data Pull — FY{FISCAL_YEAR} ===")
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    agency_df   = pull_agency_spend()
    naics_df    = pull_naics_awards()
    dist_df     = pull_award_distribution()

    print("\n=== Pull complete ===")
    print(f"  Agency records  : {len(agency_df)}")
    print(f"  NAICS awards    : {len(naics_df)}")
    print(f"  Distribution    : {len(dist_df)}")
    print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nAll files saved to: {os.path.abspath(OUTPUT_DIR)}")


=== USAspending Data Pull — FY2024 ===
Start: 2026-04-12 12:10:57

[1/3] Pulling agency spend summaries...
  → Department of Defense
  → Department of Health and Human Services
  → General Services Administration
  → Department of Energy
  → Department of Agriculture
  Attempt 1 failed: 404 Client Error: Not Found for url: https://api.usaspending.gov/api/v2/agency/021/budgetary_resources/?fiscal_year=2024
  Attempt 2 failed: 404 Client Error: Not Found for url: https://api.usaspending.gov/api/v2/agency/021/budgetary_resources/?fiscal_year=2024
  Attempt 3 failed: 404 Client Error: Not Found for url: https://api.usaspending.gov/api/v2/agency/021/budgetary_resources/?fiscal_year=2024
  → Department of Homeland Security
  Saved 5 agency records → /Users/kesh/data/agency_spend.csv

[2/3] Pulling contract awards by NAICS...
  → NAICS prefix 54...
  → NAICS prefix 33...
  → NAICS prefix 23...
  → NAICS prefix 62...
  → NAICS prefix 51...
  → NAICS prefix 48...
  Saved 600 award records → /Us

In [5]:
import requests
import pandas as pd
import os
import time

DATA_DIR = os.path.join(os.getcwd(), "data")

# Fix 1: Correct Agriculture CGAC code (012, not 021)
print("Fixing Agriculture pull...")
r = requests.get("https://api.usaspending.gov/api/v2/agency/012/budgetary_resources/",
                 params={"fiscal_year": 2024})
if r.status_code == 200:
    data = r.json()
    for yr in data.get("agency_data_by_year", []):
        if yr.get("fiscal_year") == 2024:
            print(f"  Agriculture obligations: ${yr.get('agency_total_obligated',0)/1e9:.1f}B")

# Fix 2: Correct distribution endpoint
print("\nFixing award distribution pull...")
payload = {
    "filters": {
        "time_period": [{"start_date": "2023-10-01", "end_date": "2024-09-30"}],
        "award_type_codes": ["A","B","C","D"]
    },
    "category": "awarding_agency",
    "limit": 50,
    "page": 1
}
r = requests.post("https://api.usaspending.gov/api/v2/search/spending_by_category/awarding_agency/",
                  json=payload)
print(f"  Status: {r.status_code}")
if r.status_code == 200:
    results = r.json().get("results", [])
    df = pd.DataFrame([{"agency": x.get("name",""), "amount": x.get("amount",0)} for x in results])
    df["amount_B"] = (df["amount"] / 1e9).round(2)
    df["share_pct"] = (df["amount"] / df["amount"].sum() * 100).round(2)
    df["rank"] = range(1, len(df)+1)
    df.to_csv(os.path.join(DATA_DIR, "award_distribution.csv"), index=False)
    print(f"  Saved {len(df)} distribution records")
    print(df.head())
else:
    print(f"  Error: {r.text[:200]}")

Fixing Agriculture pull...
  Agriculture obligations: $253.9B

Fixing award distribution pull...
  Status: 200
  Saved 50 distribution records
                                    agency        amount  amount_B  share_pct  \
0                    Department of Defense  4.459664e+11    445.97      60.21   
1           Department of Veterans Affairs  6.687583e+10     66.88       9.03   
2                     Department of Energy  4.604404e+10     46.04       6.22   
3  Department of Health and Human Services  2.986841e+10     29.87       4.03   
4          General Services Administration  2.647866e+10     26.48       3.57   

   rank  
0     1  
1     2  
2     3  
3     4  
4     5  


In [10]:
"""
Federal Contract Spend Analysis
Step 2: Clean and validate raw data
Author: Kesheka Edupuganti
"""

import pandas as pd
import numpy as np
import os
import json
from datetime import datetime

DATA_DIR   = os.path.join(os.getcwd(), "data")
OUTPUT_DIR = DATA_DIR

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def make_serializable(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

def clean_agency_spend():
    log("Cleaning agency_spend.csv...")
    df = pd.read_csv(os.path.join(DATA_DIR, "agency_spend.csv"))
    for col in ["total_budgetary_resources", "total_obligations", "total_outlays"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["obligation_rate_pct"] = (
        (df["total_obligations"] / df["total_budgetary_resources"])
        .replace([np.inf, -np.inf], 0).fillna(0).round(4) * 100
    )
    for col in ["total_budgetary_resources", "total_obligations", "total_outlays"]:
        df[f"{col}_B"] = (df[col] / 1e9).round(2)
    df = df[df["agency_name"].notna() & (df["agency_name"] != "")]
    df.to_csv(os.path.join(OUTPUT_DIR, "agency_spend_clean.csv"), index=False)
    log(f"  Saved {len(df)} agency rows")
    return df

def clean_naics_awards():
    log("Cleaning naics_awards_raw.csv...")
    df = pd.read_csv(os.path.join(DATA_DIR, "naics_awards_raw.csv"))
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    df["award_amount"] = pd.to_numeric(df["award_amount"], errors="coerce").fillna(0)
    df = df[df["award_amount"] > 0]
    df = df.drop_duplicates(subset=["award_id"])
    df["naics_code"] = df["naics_code"].astype(str).str.strip()
    df["naics_sector"] = df["naics_code"].str[:2]
    df["state_code"] = df["place_of_performance_state_code"].astype(str).str.upper().str.strip()
    df.loc[df["state_code"].str.len() != 2, "state_code"] = "UNKNOWN"
    def tier(amount):
        if amount >= 10_000_000:  return "A — Large (≥$10M)"
        elif amount >= 1_000_000: return "B — Mid ($1M–$10M)"
        elif amount >= 100_000:   return "C — Small ($100K–$1M)"
        else:                     return "D — Micro (<$100K)"
    df["contract_tier"] = df["award_amount"].apply(tier)
    for col in ["period_of_performance_start_date", "period_of_performance_current_end_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    if all(c in df.columns for c in ["period_of_performance_start_date", "period_of_performance_current_end_date"]):
        df["contract_duration_days"] = (
            df["period_of_performance_current_end_date"] - df["period_of_performance_start_date"]
        ).dt.days
    df.to_csv(os.path.join(OUTPUT_DIR, "naics_awards_clean.csv"), index=False)
    log(f"  Saved {len(df)} award rows")
    return df

def clean_award_distribution():
    log("Cleaning award_distribution.csv...")
    df = pd.read_csv(os.path.join(DATA_DIR, "award_distribution.csv"))
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["amount_B"] = (df["amount"] / 1e9).round(2)
    df["share_pct"] = (df["amount"] / df["amount"].sum() * 100).round(2)
    df = df.sort_values("amount", ascending=False).reset_index(drop=True)
    df["rank"] = df.index + 1
    df.to_csv(os.path.join(OUTPUT_DIR, "award_distribution_clean.csv"), index=False)
    log(f"  Saved {len(df)} distribution rows")
    return df

# ── Run all cleaning ──
agency_df = clean_agency_spend()
naics_df  = clean_naics_awards()
dist_df   = clean_award_distribution()

# ── Summary stats ──
summary = {
    "generated_at": datetime.now().isoformat(),
    "agency_spend": {
        "total_agencies": int(len(agency_df)),
        "total_obligations_B": round(float(agency_df["total_obligations_B"].sum()), 2),
        "avg_obligation_rate_pct": round(float(agency_df["obligation_rate_pct"].mean()), 2),
        "top_agency_by_obligation": agency_df.sort_values("total_obligations", ascending=False).iloc[0]["agency_name"],
    },
    "naics_awards": {
        "total_awards": int(len(naics_df)),
        "total_value_B": round(float(naics_df["award_amount"].sum() / 1e9), 2),
        "tier_distribution": {str(k): int(v) for k, v in naics_df["contract_tier"].value_counts().items()},
    },
    "award_distribution": {
        "total_agencies_ranked": int(len(dist_df)),
        "top_3_agencies": dist_df.head(3)["agency"].tolist(),
        "top_3_share_pct": round(float(dist_df.head(3)["share_pct"].sum()), 2),
    },
}

out = os.path.join(OUTPUT_DIR, "summary_stats.json")
with open(out, "w") as f:
    json.dump(summary, f, indent=2, default=make_serializable)

print("\n=== Cleaning complete ===")
print(json.dumps(summary, indent=2, default=make_serializable))

[12:23:47] Cleaning agency_spend.csv...
[12:23:47]   Saved 5 agency rows
[12:23:47] Cleaning naics_awards_raw.csv...
[12:23:47]   Saved 595 award rows
[12:23:47] Cleaning award_distribution.csv...
[12:23:47]   Saved 50 distribution rows

=== Cleaning complete ===
{
  "generated_at": "2026-04-12T12:23:47.669552",
  "agency_spend": {
    "total_agencies": 5,
    "total_obligations_B": 4178.34,
    "avg_obligation_rate_pct": 72.56,
    "top_agency_by_obligation": "Department of Health and Human Services"
  },
  "naics_awards": {
    "total_awards": 595,
    "total_value_B": 1524.33,
    "tier_distribution": {
      "A \u2014 Large (\u2265$10M)": 595
    }
  },
  "award_distribution": {
    "total_agencies_ranked": 50,
    "top_3_agencies": [
      "Department of Defense",
      "Department of Veterans Affairs",
      "Department of Energy"
    ],
    "top_3_share_pct": 75.46
  }
}


In [12]:
"""
Federal Contract Spend Analysis
Step 4: Load cleaned data into SQLite and run SQL analysis
Author: Kesheka Edupuganti
"""

import sqlite3
import pandas as pd
import os
import json

DATA_DIR   = os.path.join(os.getcwd(), "data")
OUTPUT_DIR = os.path.join(os.getcwd(), "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUTPUT_DIR, "federal_contracts.db")

def load_to_sqlite():
    conn = sqlite3.connect(DB_PATH)
    tables = {
        "agency_spend":       "agency_spend_clean.csv",
        "naics_awards":       "naics_awards_clean.csv",
        "award_distribution": "award_distribution_clean.csv",
    }
    for table, filename in tables.items():
        path = os.path.join(DATA_DIR, filename)
        if not os.path.exists(path):
            print(f"  WARNING: {filename} not found")
            continue
        df = pd.read_csv(path)
        df.to_sql(table, conn, if_exists="replace", index=False)
        print(f"  Loaded {len(df):,} rows → {table}")
    conn.close()
    print(f"  Database saved → {DB_PATH}")

def run_query(conn, label, sql):
    try:
        df = pd.read_sql_query(sql, conn)
        print(f"\n── {label} ──")
        print(df.to_string(index=False))
        return df
    except Exception as e:
        print(f"  ERROR in '{label}': {e}")
        return pd.DataFrame()

def run_all_analysis():
    conn = sqlite3.connect(DB_PATH)
    results = {}

    results["agency_obligations"] = run_query(conn, "Top Agencies by Obligation", """
        SELECT agency_name,
               ROUND(total_obligations_B, 2)  AS obligations_B,
               ROUND(obligation_rate_pct, 1)  AS obligation_rate_pct,
               CASE WHEN obligation_rate_pct >= 90 THEN 'High'
                    WHEN obligation_rate_pct >= 70 THEN 'Moderate'
                    ELSE 'Low' END AS efficiency
        FROM agency_spend
        ORDER BY total_obligations DESC
    """)

    results["naics_by_sector"] = run_query(conn, "Contract Value by NAICS Sector", """
        SELECT naics_sector,
               COUNT(*)                           AS award_count,
               ROUND(SUM(award_amount)/1e9, 2)   AS total_value_B,
               ROUND(AVG(award_amount)/1e6, 2)   AS avg_award_M
        FROM naics_awards
        GROUP BY naics_sector
        ORDER BY total_value_B DESC
    """)

    results["tier_distribution"] = run_query(conn, "Contract Tier Distribution", """
        SELECT contract_tier,
               COUNT(*)                                                          AS contract_count,
               ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM naics_awards), 1)     AS pct_count,
               ROUND(SUM(award_amount)/1e9, 2)                                  AS total_value_B
        FROM naics_awards
        GROUP BY contract_tier
        ORDER BY MIN(award_amount) DESC
    """)

    results["spend_concentration"] = run_query(conn, "Top Agencies by Spend", """
        SELECT rank, agency,
               ROUND(amount_B, 2)  AS total_B,
               ROUND(share_pct, 2) AS share_pct
        FROM award_distribution
        ORDER BY rank
        LIMIT 10
    """)

    # KPIs
    kpi_queries = {
        "total_spend_B":       "SELECT ROUND(SUM(award_amount)/1e9,1) AS v FROM naics_awards",
        "unique_contractors":  "SELECT COUNT(DISTINCT recipient_name) AS v FROM naics_awards",
        "avg_contract_M":      "SELECT ROUND(AVG(award_amount)/1e6,2) AS v FROM naics_awards",
        "tier_a_pct_of_spend": """
            SELECT ROUND(
                SUM(CASE WHEN contract_tier='A — Large (≥$10M)' THEN award_amount ELSE 0 END)
                / SUM(award_amount)*100, 1) AS v
            FROM naics_awards
        """,
    }
    kpis = {}
    print("\n── KPI Summary ──")
    for key, sql in kpi_queries.items():
        try:
            val = pd.read_sql_query(sql, conn).iloc[0, 0]
            kpis[key] = val
            print(f"  {key}: {val}")
        except Exception as e:
            print(f"  ERROR {key}: {e}")

    for key, val in results.items():
        if isinstance(val, pd.DataFrame) and not val.empty:
            val.to_csv(os.path.join(OUTPUT_DIR, f"{key}.csv"), index=False)

    with open(os.path.join(OUTPUT_DIR, "kpis.json"), "w") as f:
        json.dump({k: float(v) if v is not None else None for k, v in kpis.items()}, f, indent=2)

    print(f"\n  All results exported → {OUTPUT_DIR}")
    conn.close()

# ── Run ──
print("=== SQL Analysis Pipeline ===")
print("Loading data into SQLite...")
load_to_sqlite()
print("\nRunning analysis queries...")
run_all_analysis()
print("\n=== Analysis complete ===")

=== SQL Analysis Pipeline ===
Loading data into SQLite...
  Loaded 5 rows → agency_spend
  Loaded 595 rows → naics_awards
  Loaded 50 rows → award_distribution
  Database saved → /Users/kesh/output/federal_contracts.db

Running analysis queries...

── Top Agencies by Obligation ──
                            agency_name  obligations_B  obligation_rate_pct efficiency
Department of Health and Human Services        2518.65                 87.9   Moderate
                  Department of Defense        1393.42                 70.0        Low
        Department of Homeland Security         140.63                 79.8   Moderate
                   Department of Energy          81.73                 53.3        Low
        General Services Administration          43.91                 71.8   Moderate

── Contract Value by NAICS Sector ──
naics_sector  award_count  total_value_B  avg_award_M
        None          595        1524.33       2561.9

── Contract Tier Distribution ──
    contract_tie